Install dependencies

In [ ]:

# BAYORA BLUE TEAM - STABLE DEPENDENCIES

%pip install -q \
    "transformers==4.46.3" \


    "datasets==3.1.0" \
    "peft==0.13.2" \
    "accelerate==1.1.1" \
    "scikit-learn>=1.6,<2.0"


In [ ]:

variables = [
    "raw_ds",
    "df",
    "train_df",
    "val_df",
    "test_df",
    "dataset_dict",
    "tokenizer",
    "tokenized_datasets",
    "data_collator",
    "model",
    "lora_config",
    "training_args",
    "trainer"
]

for variable in variables:
    print(f"{variable}: {'Available' if variable in globals() else 'Missing'}")


raw_ds: Missing
df: Missing
train_df: Missing
val_df: Missing
test_df: Missing
dataset_dict: Missing
tokenizer: Missing
tokenized_datasets: Missing
data_collator: Missing
model: Missing
lora_config: Missing
training_args: Missing
trainer: Missing


In [ ]:

import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory:", round(
        torch.cuda.get_device_properties(0).total_memory / 1024**3, 2
    ), "GB")


CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB


In [ ]:

import torch
import os

print("========== ENVIRONMENT STATUS ==========")

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: GPU not detected.")

print("\n========== VARIABLES ==========")

required = [
    "raw_ds",
    "df",
    "dataset_dict",
    "tokenizer",
    "tokenized_datasets",
    "data_collator",
    "model",
    "training_args",
    "trainer"
]

for name in required:
    print(f"{name}: {'Available' if name in globals() else 'Missing'}")

print("\n========== GOOGLE DRIVE ==========")
print("Drive mounted:", os.path.exists("/content/drive/MyDrive"))


========== ENVIRONMENT STATUS ==========
PyTorch: 2.14.1+cu130
CUDA available: True
GPU: Tesla T4

========== VARIABLES ==========
raw_ds: Missing
df: Missing
dataset_dict: Missing
tokenizer: Missing
tokenized_datasets: Missing
data_collator: Missing
model: Missing
training_args: Missing
trainer: Missing

========== GOOGLE DRIVE ==========
Drive mounted: False


In [ ]:

import torch

print("GPU:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))

print("\nVariable status:")

for name in [
    "raw_ds", "df", "dataset_dict",
    "tokenizer", "tokenized_datasets",
    "data_collator", "model",
    "training_args", "trainer"
]:
    print(name, "->", "Available" if name in globals() else "Missing")


GPU: True
GPU Name: Tesla T4

Variable status:
raw_ds -> Missing
df -> Missing
dataset_dict -> Missing
tokenizer -> Missing
tokenized_datasets -> Missing
data_collator -> Missing
model -> Missing
training_args -> Missing
trainer -> Missing


In [ ]:

from google.colab import drive
import os

drive.mount("/content/drive")

BASE_DIR = "/content/drive/MyDrive/bayora"
OUTPUT_DIR = os.path.join(BASE_DIR, "blue_agent_results")
SAVE_PATH = os.path.join(BASE_DIR, "blue_agent_lora")
DATA_DIR = os.path.join(BASE_DIR, "datasets")

for path in [OUTPUT_DIR, SAVE_PATH, DATA_DIR]:
    os.makedirs(path, exist_ok=True)

print("Google Drive connected!")
print("Model save path:", SAVE_PATH)
print("Results path:", OUTPUT_DIR)


Mounted at /content/drive
Google Drive connected!
Model save path: /content/drive/MyDrive/bayora/blue_agent_lora
Results path: /content/drive/MyDrive/bayora/blue_agent_results


In [ ]:

from datasets import load_dataset

print("Loading ToxiGen dataset...")

raw_ds = load_dataset(
    "skg/toxigen-data",
    name="train",
    split="train"
)

print("Dataset loaded!")
print("Total examples:", len(raw_ds))
print("Columns:", raw_ds.column_names)


Loading ToxiGen dataset...


README.md:   0%|          | 0.00/10.7k [00:00<?, ?B/s]

train/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 18.8MB            

train/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/250951 [00:00<?, ? examples/s]

Dataset loaded!
Total examples: 250951
Columns: ['prompt', 'generation', 'generation_method', 'group', 'prompt_label', 'roberta_prediction']


In [ ]:

import pandas as pd
import numpy as np

df = raw_ds.to_pandas()

# Remove invalid generated responses
df = df[
    df["generation"].notna()
    & df["roberta_prediction"].notna()
    & (df["generation"].str.strip() != "")
].copy()

# Assign conservative pseudo-labels
df["label"] = np.select(
    [
        df["roberta_prediction"] <= 0.10,
        df["roberta_prediction"] >= 0.90
    ],
    [0, 1],
    default=-1
)

# Remove uncertain examples
df = df[df["label"] != -1].copy()

# Prepare final columns
df = df[["generation", "label", "group", "roberta_prediction"]]
df.rename(columns={"generation": "text"}, inplace=True)

df.drop_duplicates(subset=["text"], inplace=True)
df.reset_index(drop=True, inplace=True)

# Save recovery copy
df.to_csv(
    os.path.join(DATA_DIR, "toxigen_weak_labels.csv"),
    index=False
)

print("Dataset preparation completed!")
print("Total examples:", len(df))
print("\nClass distribution:")
print(df["label"].value_counts().sort_index())
print("\nClass percentages:")
print((df["label"].value_counts(normalize=True) * 100).round(2))
print("\nSaved to Google Drive.")


Dataset preparation completed!
Total examples: 210614

Class distribution:
label
0    165409
1     45205
Name: count, dtype: int64

Class percentages:
label
0    78.54
1    21.46
Name: proportion, dtype: float64

Saved to Google Drive.


In [ ]:

assert len(df) > 0
assert df["text"].notna().all()
assert set(df["label"].unique()) == {0, 1}

counts = df["label"].value_counts()

assert counts.min() >= 2, "Insufficient examples in one class."

print("Validation passed!")
print("Non-toxic:", counts[0])
print("Toxic:", counts[1])


Validation passed!
Non-toxic: 165409
Toxic: 45205


In [ ]:

from sklearn.model_selection import train_test_split

# 80% training, 20% temporary
train_df, temp_df = train_test_split(
    df,
    test_size=0.20,
    random_state=42,
    stratify=df["label"]
)

# 10% validation, 10% testing
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df["label"]
)

print("Training:", len(train_df))
print("Validation:", len(val_df))
print("Testing:", len(test_df))

print("\nClass distribution:")
for name, part in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Test", test_df)
]:
    print(f"\n{name}:")
    print(part["label"].value_counts())

# Save splits for recovery
train_df.to_csv(os.path.join(DATA_DIR, "train.csv"), index=False)
val_df.to_csv(os.path.join(DATA_DIR, "validation.csv"), index=False)
test_df.to_csv(os.path.join(DATA_DIR, "test.csv"), index=False)

print("\nAll splits saved to Google Drive.")


Training: 168491
Validation: 21061
Testing: 21062

Class distribution:

Train:
label
0    132327
1     36164
Name: count, dtype: int64

Validation:
label
0    16541
1     4520
Name: count, dtype: int64

Test:
label
0    16541
1     4521
Name: count, dtype: int64

All splits saved to Google Drive.


In [ ]:

from datasets import Dataset, DatasetDict

train_dataset = Dataset.from_pandas(
    train_df[["text", "label"]].reset_index(drop=True)
)

val_dataset = Dataset.from_pandas(
    val_df[["text", "label"]].reset_index(drop=True)
)

test_dataset = Dataset.from_pandas(
    test_df[["text", "label"]].reset_index(drop=True)
)

dataset_dict = DatasetDict({
    "train": train_dataset,
    "validation": val_dataset,
    "test": test_dataset
})

print(dataset_dict)


DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 168491
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 21061
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 21062
    })
})


Load Toxic-BERT tokenizer

In [ ]:

from transformers import AutoTokenizer

MODEL_NAME = "unitary/toxic-bert"
MAX_LENGTH = 256

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Tokenizer loaded successfully!")
print("Vocabulary size:", tokenizer.vocab_size)
print("Maximum length:", MAX_LENGTH)


config.json:   0%|          | 0.00/811 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/174 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Tokenizer loaded successfully!
Vocabulary size: 30522
Maximum length: 256


Tokenize datasets

In [ ]:

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_LENGTH
    )

tokenized_datasets = dataset_dict.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"],
    desc="Tokenizing datasets"
)

print(tokenized_datasets)


Tokenizing datasets:   0%|          | 0/168491 [00:00<?, ? examples/s]

Tokenizing datasets:   0%|          | 0/21061 [00:00<?, ? examples/s]

Tokenizing datasets:   0%|          | 0/21062 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 168491
    })
    validation: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 21061
    })
    test: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 21062
    })
})


Configure dynamic padding

In [ ]:

from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    pad_to_multiple_of=8
)

print("Dynamic padding configured!")


Dynamic padding configured!


Load Toxic-BERT

In [ ]:

import torch
from transformers import AutoModelForSequenceClassification

assert torch.cuda.is_available(), "GPU not detected!"

print("GPU:", torch.cuda.get_device_name(0))

id2label = {
    0: "NON_TOXIC",
    1: "TOXIC"
}

label2id = {
    "NON_TOXIC": 0,
    "TOXIC": 1
}

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label=id2label,
    label2id=label2id,
    problem_type="single_label_classification",
    ignore_mismatched_sizes=True
)

print("\nToxic-BERT loaded successfully!")
print("Number of labels:", model.config.num_labels)


[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `6`.


GPU: Tesla T4


model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: unitary/toxic-bert
Key               | Status   |                                                                                       
------------------+----------+---------------------------------------------------------------------------------------
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([6]) vs model:torch.Size([2])          
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([6, 768]) vs model:torch.Size([2, 768])

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.



Toxic-BERT loaded successfully!
Number of labels: 2


Configure LoRA

In [ ]:

%pip uninstall -y torchao


Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [ ]:

import importlib.metadata

try:
    print("TorchAO version:", importlib.metadata.version("torchao"))
except importlib.metadata.PackageNotFoundError:
    print("TorchAO successfully removed.")

import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))


TorchAO successfully removed.
CUDA available: True
GPU: Tesla T4


In [ ]:

from peft import LoraConfig, TaskType, get_peft_model

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["query", "value"],
    modules_to_save=["classifier"],
    bias="none"
)

model = get_peft_model(model, lora_config)

print("LoRA configured successfully!")
model.print_trainable_parameters()


LoRA configured successfully!
trainable params: 296,450 || all params: 109,780,228 || trainable%: 0.2700


In [ ]:

import torch

model = model.to("cuda")

print("Model device:", next(model.parameters()).device)
print("GPU:", torch.cuda.get_device_name(0))

print(
    "GPU memory allocated:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)

assert next(model.parameters()).is_cuda
print("\nGPU verification passed!")


Model device: cuda:0
GPU: Tesla T4
GPU memory allocated: 0.41 GB

GPU verification passed!


In [ ]:

import numpy as np
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    roc_auc_score
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)
    probabilities = torch.softmax(
        torch.tensor(logits), dim=-1
    ).numpy()[:, 1]

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="binary",
        zero_division=0
    )

    return {
        "accuracy": accuracy_score(labels, predictions),
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "roc_auc": roc_auc_score(labels, probabilities)
    }

print("Evaluation metrics configured!")


Evaluation metrics configured!


Training configuration

In [ ]:

import inspect
from transformers import TrainingArguments

OUTPUT_DIR = "/content/drive/MyDrive/bayora/blue_agent_results"

params = inspect.signature(TrainingArguments.__init__).parameters

training_kwargs = {
    "output_dir": OUTPUT_DIR,
    "num_train_epochs": 3,
    "learning_rate": 2e-4,
    "per_device_train_batch_size": 8,
    "per_device_eval_batch_size": 16,
    "gradient_accumulation_steps": 2,
    "weight_decay": 0.01,
    "logging_steps": 100,
    "save_strategy": "epoch",
    "load_best_model_at_end": True,
    "metric_for_best_model": "f1",
    "greater_is_better": True,
    "save_total_limit": 2,
    "fp16": True,
    "report_to": "none",
    "dataloader_num_workers": 2,
    "seed": 42
}

# Add optional arguments only when supported
if "warmup_ratio" in params:
    training_kwargs["warmup_ratio"] = 0.1
elif "warmup_steps" in params:
    training_kwargs["warmup_steps"] = 100

if "eval_strategy" in params:
    training_kwargs["eval_strategy"] = "epoch"
elif "evaluation_strategy" in params:
    training_kwargs["evaluation_strategy"] = "epoch"

training_args = TrainingArguments(**training_kwargs)

print("Training configuration ready!")
print("Transformers version compatibility checked.")
print("GPU mixed precision:", training_args.fp16)


Training configuration ready!
Transformers version compatibility checked.
GPU mixed precision: True


In [ ]:

print("Model exists:", "model" in globals())
print("Training arguments exist:", "training_args" in globals())
print("Tokenized dataset exists:", "tokenized_datasets" in globals())
print("Data collator exists:", "data_collator" in globals())
print("Metrics function exists:", "compute_metrics" in globals())


Model exists: True
Training arguments exist: True
Tokenized dataset exists: True
Data collator exists: True
Metrics function exists: True


In [ ]:

from transformers import Trainer, EarlyStoppingCallback
import inspect

trainer_kwargs = {
    "model": model,
    "args": training_args,
    "train_dataset": tokenized_datasets["train"],
    "eval_dataset": tokenized_datasets["validation"],
    "data_collator": data_collator,
    "compute_metrics": compute_metrics,
    "callbacks": [EarlyStoppingCallback(early_stopping_patience=2)]
}

trainer_params = inspect.signature(Trainer.__init__).parameters

if "processing_class" in trainer_params:
    trainer_kwargs["processing_class"] = tokenizer
else:
    trainer_kwargs["tokenizer"] = tokenizer

trainer = Trainer(**trainer_kwargs)

print("Trainer initialized successfully!")


Trainer initialized successfully!


In [ ]:

import torch

assert torch.cuda.is_available()
assert next(trainer.model.parameters()).is_cuda

print("GPU:", torch.cuda.get_device_name(0))
print("Model device:", next(trainer.model.parameters()).device)
print("Ready to train!")


GPU: Tesla T4
Model device: cuda:0
Ready to train!


Start training

In [ ]:

import time

print("Starting Blue Team Toxicity Classifier training...")
start_time = time.time()

train_result = trainer.train()

end_time = time.time()

print("\nTraining completed!")
print("Training duration:", round((end_time - start_time) / 60, 2), "minutes")
print("Training loss:", train_result.training_loss)


Starting Blue Team Toxicity Classifier training...


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Roc Auc
1,0.970835,0.520610,0.785385,0.000000,0.000000,0.000000,0.498017
